# 07 — Telephony — Telnyx

Call Control, Ed25519, AMD, DTMF, track filter, anti-replay.


## Optional: run in Docker

Uncomment the cell below to launch JupyterLab + EmbeddedServer in a container. It's idempotent and a no-op once you're already inside the container. Container helpers for the TypeScript notebooks are tracked separately — see issue #80 for status.


In [ ]:
// Optional — launch the patter-notebooks Docker stack from this cell.
// Skip this cell to run on your host runtime. Idempotent if uncommented.
//
// import * as _setup from "./_setup.ts";
// await _setup.startDocker();             // build + up -d, prints http://localhost:8888
// await _setup.startDocker({ openUrl: true });  // …and open the browser tab


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises Telnyx carrier construction and Ed25519 webhook signature verification.


### Telnyx carrier construction


In [ ]:
import { Patter, Telnyx } from "getpatter";
await cell('telnyx_carrier', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Telnyx({ apiKey: 'KEY0_TEST_TELNYX', publicKey: '' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Telnyx carrier constructed OK');
});


### Ed25519 sign + verify


In [ ]:
import { webcrypto } from 'crypto';
await cell('ed25519_verify', { tier: 1, env }, async () => {
  const { subtle } = webcrypto;
  const keypair = await subtle.generateKey({ name: 'Ed25519' }, true, ['sign', 'verify']);
  const payload = new TextEncoder().encode('telnyx-webhook-payload');
  const signature = await subtle.sign('Ed25519', keypair.privateKey, payload);
  const valid = await subtle.verify('Ed25519', keypair.publicKey, signature, payload);
  console.log(`Ed25519 sign+verify: ${valid}  sig_len=${signature.byteLength}`);
  const tampered = new TextEncoder().encode('tampered');
  const invalidOk = !(await subtle.verify('Ed25519', keypair.publicKey, signature, tampered));
  console.log(`Tampered payload rejected: ${invalidOk}`);
});


### Telnyx anti-replay: timestamp window check


In [ ]:
await cell('telnyx_timestamp', { tier: 1, env }, () => {
  const WINDOW = 300;
  const now = Math.floor(Date.now() / 1000);
  const cases: [number, string][] = [
    [now,       'fresh'],
    [now - 60,  'recent'],
    [now - 301, 'stale — reject'],
  ];
  for (const [ts, label] of cases) {
    const age = Math.abs(now - ts);
    const ok = age <= WINDOW ? '✓' : '✗';
    console.log(`  ${ok} age=${age}s  ${label}`);
  }
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a real call through the Telnyx carrier. Requires `ENABLE_LIVE_CALLS=1` and Telnyx credentials.


### Pre-flight checklist


In [ ]:
await cell('live_preflight', { tier: 4, required: ['TELNYX_API_KEY', 'TELNYX_CONNECTION_ID', 'TELNYX_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'], env }, () => {
  console.log(`  carrier:       Telnyx ${env.telnyxNumber}  →  ${env.targetNumber}`);
  console.log(`  connection_id: ${env.telnyxConnectionId}`);
  console.log(`  webhook:       ${env.publicWebhookUrl || '(ngrok auto-launch)'}`);
});


### Live Telnyx outbound call *(T4)*


In [ ]:
import { Patter, Telnyx, OpenAIRealtime } from "getpatter";
await cell('live_telnyx_call', { tier: 4, required: ['TELNYX_API_KEY', 'TELNYX_CONNECTION_ID', 'TELNYX_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, async () => {
  const p = new Patter({
    carrier: new Telnyx({ apiKey: env.telnyxKey, publicKey: env.telnyxPublicKey }),
    phoneNumber: env.telnyxNumber,
    webhookUrl: env.publicWebhookUrl,
  });
  const agent = p.agent({
    systemPrompt: 'Telnyx demo agent. Greet and hang up.',
    engine: new OpenAIRealtime({ apiKey: env.openaiKey }),
  });
  try {
    await p.call(env.targetNumber, { agent, firstMessage: 'Hello from Patter via Telnyx.', ringTimeout: env.maxCallSeconds });
    console.log('✓ Telnyx call completed');
  } finally {
    await hangupLeftoverCalls(env);
  }
});
